## 1、获取大模型

In [3]:
import dotenv
from langchain.chains.summarize.map_reduce_prompt import prompt_template
from langchain_openai import ChatOpenAI
import os

# dotenv.load_dotenv()
# os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
# os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
#
# llm = ChatOpenAI(model="GPT-4o-mini")
#
# response = llm.invoke("什么是大模型？")
# print(response)

from langchain_ollama import ChatOllama

llm = ChatOllama(base_url="http://localhost:11434", model="qwen3.5:2b", reasoning=False)
print(llm.invoke("什么是LangChain?"))


content='**LangChain** 是一个由 AI 领域研究者构建的**开源框架**，旨在简化与大型语言模型（LLMs）的交互。它的核心目标是让开发者能够快速构建**基于 LLM 的应用程序（RAG 应用、Agent 系统、Chat 机器人等）**，而不必从底层手动处理提示词（Prompt Engineering）和复杂的计算逻辑。\n\n简单来说，LangChain 提供了一套**高级编程接口（API）**和**最佳实践模板**，将构建 AI 应用的繁琐工作自动化。\n\n---\n\n### 核心功能与优势\n\n1.  **简化 Prompt 工程**\n    *   开发者只需关注业务逻辑，将复杂的 Prompt 设计交给 LangChain 自动优化。\n    *   例如，利用 LangSmith（内部监控工具）追踪 Prompt 如何被调用、模型如何响应。\n\n2.  **丰富的 Agent 能力**\n    *   内置强大的**智能体**（Agents）库，允许构建具备自主决策能力的 AI 机器人，能自动搜索网络、规划任务、执行代码等，而无需手动编写每一步的指令。\n\n3.  **RAG 支持（检索增强生成）**\n    *   专为解决“幻觉”问题而设计。它允许将外部知识库（如 PDF、网页、数据库）与 LLM 结合，让 AI 在回答时优先参考用户提供的上下文信息，而非凭空编造。\n\n4.  **丰富的生态与模板**\n    *   支持 **Python** 和 **JavaScript (TypeScript)** 两种语言。\n    *   提供大量现成的“开箱即用”的应用程序（Apps）模板，涵盖了从聊天机器人到代码生成工具等场景。\n\n5.  **监控与调试**\n    *   内置 **LangSmith**，可以可视化地监控 Prompt、Agent 行为、模型推理时间以及输出结果，帮助排查问题。\n\n---\n\n### 典型应用场景\n\n*   **智能客服聊天机器人**：理解用户自然语言，回答问题并推荐产品。\n*   **代码生成与调试助手**：读取代码库，自动修复 Bug 或生成新功能。\n*   **智能数据分析师**：结合 RAG，让用户通过对话直接查询企业内部的报表数据，而不需要人

## 2、使用提示词模版

In [16]:
from langchain_core.prompts import ChatPromptTemplate

# 需要注意一点，这里需要指明具体的 role，在这里是 system 和用户
prompt = ChatPromptTemplate.from_messages([
    ("system","你是世界级的技术文档编写者"),
    ("user","{input}")
])

# 我们可以把 prompt 和 llm 的调用合在一起
chain = prompt | llm
message = chain.invoke({"input": "大模型中的LangChain 是什么？"})

print(message)

content='# LangChain 概述\n\n**LangChain** 是由 LangChain LLC（现更名为 LangChain Inc.）开发的一个**基于框架的编程库**，专为构建基于链式的复杂应用而设计。它旨在简化与大语言模型（LLM）交互、编排工作流以及处理复杂任务的开发过程。\n\n---\n\n## 📌 核心定位\n\nLangChain 并非一个独立的 AI 模型，而是一个**工程框架**。它通过抽象和标准化大模型的使用方式，让开发者能够：\n\n- ✅ 将大模型能力融入现有工作流（Workflow）\n- ✅ 实现模块化架构，便于维护与扩展\n- ✅ 统一调用多种 LLM 模型（如 OpenAI、LocalLLaMA 等）\n- ✅ 自动化提示词工程、任务规划和数据清洗\n- ✅ 支持 RAG（检索增强生成）、Agent（智能体）、CoT（思维链）等高级模式\n\n---\n\n## 🔑 关键特性\n\n### 1. **链式编程（Chain Pattern）**\n使用 `chain` 将多个组件串联（如：读取文档 → 向量检索 → 提示词生成 → 调用模型 → 后处理），形成可复用的逻辑流。\n\n```python\nfrom langchain_community.llms import AzureChatOpenAI\nfrom langchain_community.chat_models import ChatAzureChatOpenAI\n\n# 示例：构建一个“查询天气链”\ndef query_weather(lat, long):\n    llm = AzureChatOpenAI(api_key="YOUR_API_KEY", api_version="2023-05-15")\n    chain = (\n        llm\n        .with_structured_output(dict)  # 强制输出为字典格式\n        .with_retries(3)               # 添加重试机制\n        .with_stop_sequences(["Done"])  # 添加停止词\n        .with_system_prompt("你是一个专业的天气助

## 3、使用输出解析器

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser

# 初始化模型
# llm = ChatOpenAI(model="GPT-4o-mini")
llm = ChatOllama(base_url="http://localhost:11434", model="qwen3.5:2b", reasoning=False)

prompt = ChatPromptTemplate.from_messages([
    ("system","你是世界级的技术文档编写者"),
    ("user","{input}")
])

# 使用输出解析器
# output_parser = StrOutputParser()
output_parser = JsonOutputParser()

# 将其添加到上一个链中
chain = prompt | llm | output_parser

# 调用它并提出同样的问题，答案是一个字符串，而不是 ChatMessage
message = chain.invoke({"input": "大模型中的LangChain 是什么？用 JSON 格式回复，问题用 question，回答用 answer"})

print(message)

{'question': '大模型中的 LangChain 是什么？', 'answer': 'LangChain 是一个旨在简化与构建大型语言模型（LLM）应用的工具框架。它通过提供一套标准化的 API、丰富的示例代码库、数据源连接器以及自动化编排功能，帮助开发者快速构建智能对话机器人、代码生成助手、多轮问答系统以及其他各类基于大模型的应用程序。其核心理念是‘编排（Orchestration）’，允许开发者用自然语言或简单代码定义复杂的处理流程，从而提升开发效率并降低对特定 LLM 调优知识的专业依赖。'}


## 4、使用向量存储

In [19]:
# 导入和使用WebBaseLoader
from langchain_community.document_loaders import WebBaseLoader
import bs4

# 1. 加载网页文档
loader = WebBaseLoader(
    web_path="https://www.gov.cn/yaowen/liebiao/202603/content_7063799.htm",
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="UCAP-CONTENT"))
)
docs = loader.load()
# print(docs)

# 对于嵌入大模型，这里通过 API 调用
from langchain_openai import OpenAIEmbeddings
# embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

# 2. 替换为 Ollama 本地向量模型（关键！）
from langchain_ollama import OllamaEmbeddings  # 用这个
# 本地 Ollama 嵌入模型，完全替代 text-embedding-ada-002
embeddings = OllamaEmbeddings(
    base_url="http://localhost:11434",
    model="embeddinggemma"  # 你刚 pull 的模型
)

# 3. 文档分割，使用分割器分割文档
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)
print("文档切分数量：", len(documents))

# 4. 构建 FAISS 向量库
# 向量存储 embeddings 会将 documents 中的每个文本存成向量，并将这些向量存储到 FAISS 向量数据库中
from langchain_community.vectorstores import FAISS
vector = FAISS.from_documents(documents, embeddings)
print("✅ 向量库构建完成！")
print(vector)

文档切分数量： 5
✅ 向量库构建完成！


## 5、RAG（检索增强生成）

In [24]:
from langchain_core.prompts import PromptTemplate

retriever = vector.as_retriever()
retriever.search_kwargs = {"k" : 5}
docs = retriever.invoke("五年规划是什么？")

for i, doc in enumerate(docs):
    print(f"🌟第{i+1}条规则：")
    print(doc)

# 定义提示词模版
prompt_template_custom = """
你是一个问答机器人。
你的任务是根据下述给定的己知信息回答用户问题。
确保你的回复完全依据下述已知信息。不要编造答案。
如果下述已知信息不足以回答用户的问题，请直接回复"我无法回答您的问题"。

已知信息:
{info}

用户问:{question}

请用中文回答用户问题。
"""

# 得到提示词模版对象
template = PromptTemplate.from_template(prompt_template_custom)

prompt = template.format(info=docs, question="五年规划是什么？")

# 调用 llm
response = llm.invoke(prompt)
print(response.content)

🌟第1条规则：
page_content='黄玥、赵鸿宇全国两会闭幕不久，国家第十五个五年规划纲要发布后，习近平总书记首次赴地方考察，前往河北雄安新区调研。察看启动区建设进展，考察中国华能集团有限公司、北京四中雄安校区，主持召开深入推进雄安新区高质量建设和发展座谈会……总书记雄安之行，蕴含指引“十五五”开好局起好步的深刻考量。第一，定了就要落实，一步一步朝前走。规划已定，重在落实。习近平总书记在全国两会上强调“好规划和执行力，这两条要结合起来”。9年间，从一片地、一张图到一座城，雄安新区按照既定规划部署稳扎稳打、拔节生长，“取得重大阶段性成果”，正是对这一要求的充分实践。9年间，习近平总书记四次考察雄安，一次次在关键节点作出具有针对性的谋划指引，步步推进，不断深化。战略规划循序渐进，贯彻执行一以贯之。五年规划和雄安新区建设，均体现出中国独特的制度优势和治理智慧。5年前，雄安新区首次写入国家五年规划，“十四五”规划纲要提出，“高标准高质量建设雄安新区，加快启动区和起步区建设，推动管理体制创新”。在今年3月发布的“十五五”规划纲要中，“高标准高质量推进雄安新区建设现代化城市，完善管理体制”清晰在列。两份规划，都将雄安新区纳入区域发' metadata={'source': 'https://www.gov.cn/yaowen/liebiao/202603/content_7063799.htm'}
🌟第2条规则：
page_content='高质量发展在雄安新区的生动注脚。这次的考察点与高质量发展息息相关。“中国星网、中国中化、中国华能总部，83家央企子公司有序迁移注册，这是将来雄安产业发展的基本底气。”“高校建设更是具备无限潜能，将推动这里产学研一体化发展。”习近平总书记思虑深远。此次赴雄安，从“更加有力有序推进北京非首都功能疏解和承接”，到“一体抓好高质量建设和高效能治理”，从“建设天蓝、地绿、水清的美丽雄安”，到“因地制宜发展新质生产力，培育符合新区实际的现代化产业体系”……总书记从方方面面为雄安新区高质量建设和发展指明新路径。革故鼎新，高瞻远瞩。以高质量发展推进中国式现代化，没有现成答案。雄安新区的实践，不是简单复制深圳、浦东，而是以新发展理念的深刻变革，探索可操作可推广的制度创新和发展范式，为中国的未来探路。第三，始终聚焦于人，发展与民生相统一。五年规划之所以